In [ ]:
# ==============================================================================
# TESTE 1 - INTELIGÊNCIA ARTIFICIAL
# Aluno: Pedro Sérgio Ribeiro Santana Bezerra Rodrigues
# Objetivo: Modelo de Regressão Linear para estimativa de valor de empréstimo
# ==============================================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------------------
# ETAPA 1: CARREGAR OS DADOS
# ------------------------------------------------------------------------------
print("=== ETAPA 1: CARREGAR OS DADOS ===")
try:
  from google.colab import files

  print("Por favor, selecione e faça o upload do arquivo 'base_credito.csv':")
  uploaded = files.upload()
except Exception:
  pass

df = pd.read_csv('base_credito.csv')
print("\nPrimeiros 5 registros da tabela:")
display(df.head())

# ------------------------------------------------------------------------------
# ETAPA 2: CONHECER OS DADOS
# ------------------------------------------------------------------------------
print("\n" + "=" * 50)
print("=== ETAPA 2: CONHECER OS DADOS ===")
print("Informações estruturais e tipos de dados:")
df.info()

print("\nEstatísticas Descritivas:")
display(df.describe())

print("""
Mapeamento das Variáveis:
  - Entrada (Features):
      1. idade (int64): Idade do cliente em anos.
      2. renda_mensal (float64): Renda mensal do cliente em R$.
      3. score_credito (int64): Score de crédito do cliente (300 a 850).
      4. historico_pagamentos (float64): Taxa de pontualidade nos pagamentos (0 a 100%).
  - Saída (Target):
      1. valor_emprestimo (float64): Valor do empréstimo concedido/aprovado em R$.
""")

# ------------------------------------------------------------------------------
# ETAPA 3: ANALISAR OS DADOS
# ------------------------------------------------------------------------------
print("=" * 50)
print("=== ETAPA 3: ANALISAR OS DADOS ===")

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Renda Mensal vs Valor do Empréstimo
sns.scatterplot(
    data=df.sample(2000, random_state=42),
    x='renda_mensal',
    y='valor_emprestimo',
    alpha=0.5,
    ax=axes[0],
    color='#1f77b4',
)
axes[0].set_title('Renda Mensal vs. Valor do Empréstimo (Amostra de 2.000)')
axes[0].set_xlabel('Renda Mensal (R$)')
axes[0].set_ylabel('Valor do Empréstimo (R$)')

# Gráfico 2: Score de Crédito vs Valor do Empréstimo
sns.scatterplot(
    data=df.sample(2000, random_state=42),
    x='score_credito',
    y='valor_emprestimo',
    alpha=0.5,
    ax=axes[1],
    color='#ff7f0e',
)
axes[1].set_title('Score de Crédito vs. Valor do Empréstimo (Amostra de 2.000)')
axes[1].set_xlabel('Score de Crédito')
axes[1].set_ylabel('Valor do Empréstimo (R$)')

plt.tight_layout()
plt.show()

print("\nMatriz de Correlação:")
display(df.corr())
print(
    "\nComportamento Observado: Existe uma forte relação linear positiva (r ="
    " 0.887) entre a renda mensal e o valor do empréstimo. Clientes com maior"
    " renda obtêm limites proporcionalmente maiores."
)

# ------------------------------------------------------------------------------
# ETAPA 4: PREPARAR OS DADOS
# ------------------------------------------------------------------------------
print("\n" + "=" * 50)
print("=== ETAPA 4: PREPARAR OS DADOS ===")

X = df[['idade', 'renda_mensal', 'score_credito', 'historico_pagamentos']]
y = df['valor_emprestimo']

# Divisão: 80% Treino e 20% Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Conjunto de Treinamento: {len(X_train)} amostras (80%)")
print(f"Conjunto de Teste:       {len(X_test)} amostras (20%)")

# ------------------------------------------------------------------------------
# ETAPA 5: TREINAR O MODELO
# ------------------------------------------------------------------------------
print("\n" + "=" * 50)
print("=== ETAPA 5: TREINAR O MODELO ===")

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

coef_df = pd.DataFrame(
    {'Atributo': X.columns, 'Coeficiente (Peso)': model.coef_}
)
print("Coeficientes aprendidos pelo modelo:")
display(coef_df)
print(f"Intercepto (Termo constante): {model.intercept_:.2f}")

# ------------------------------------------------------------------------------
# ETAPA 6: AVALIAR O MODELO
# ------------------------------------------------------------------------------
print("\n" + "=" * 50)
print("=== ETAPA 6: AVALIAR O MODELO ===")

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  (Erro Médio Absoluto): R$ {mae:.2f}")
print(f"RMSE (Raiz do Erro Quadrático Médio): R$ {rmse:.2f}")
print(f"R²   (Coeficiente de Determinação): {r2:.4f} ({r2*100:.2f}%)")

print("""
Interpretação das Métricas:
  - MAE (R$ 2.985,94): Em média, as estimativas do modelo erram por aproximadamente R$ 2.985,94.
  - R² (92,23%): O modelo consegue explicar 92,23% da variação no valor dos empréstimos.
""")

# ------------------------------------------------------------------------------
# ETAPA 7: COMPARAR VALORES REAIS E PREVISTOS
# ------------------------------------------------------------------------------
print("=" * 50)
print("=== ETAPA 7: COMPARAR VALORES REAIS E PREVISTOS ===")

df_comparacao = pd.DataFrame({
    'Valor Real (R$)': y_test.values[:10],
    'Valor Previsto (R$)': y_pred[:10],
    'Diferença (R$)': y_test.values[:10] - y_pred[:10],
    'Erro Relativo (%)': np.abs(
        (y_test.values[:10] - y_pred[:10]) / y_test.values[:10]
    )
    * 100,
})

print("\nExemplo comparativo ( Primeiras 10 amostras do teste ):")
display(df_comparacao)

# Gráfico de comparação
plt.figure(figsize=(7, 6))
plt.scatter(y_test[:2000], y_pred[:2000], alpha=0.3, color='#2ca02c')
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--',
    lw=2,
    label='Previsão Perfeita',
)
plt.title('Valores Reais vs. Valores Previstos (Conjunto de Teste)')
plt.xlabel('Valor Real do Empréstimo (R$)')
plt.ylabel('Valor Previsto do Empréstimo (R$)')
plt.legend()
plt.show()

# ------------------------------------------------------------------------------
# ETAPA 8: FAZER UMA PREVISÃO PARA NOVO CLIENTE
# ------------------------------------------------------------------------------
print("=" * 50)
print("=== ETAPA 8: FAZER UMA PREVISÃO ===")

novo_cliente = pd.DataFrame(
    [{
        'idade': 35,
        'renda_mensal': 10000.00,
        'score_credito': 750,
        'historico_pagamentos': 90.00,
    }]
)

valor_estimado = model.predict(novo_cliente)[0]

print("Perfil do novo cliente:")
display(novo_cliente)
print(
    f"\n>>> VALOR ESTIMADO DO EMPRÉSTIMO: R$ {valor_estimado:.2f} <<<\n"
)
print(
    "Interpretação: Um cliente de 35 anos com renda mensal de R$ 10.000,00 e"
    " score alto (750) recebe uma estimativa de R$ 45.329,08, o que equivale a"
    " ~4.5 vezes sua renda mensal."
)

# ------------------------------------------------------------------------------
# ETAPA 9: CONCLUSÃO
# ------------------------------------------------------------------------------
print("=" * 50)
print("=== ETAPA 9: CONCLUSÃO E LIMITAÇÕES ===")
print("""
Conclusão:
  O modelo de Regressão Linear apresentou excelente capacidade preditiva (R² = 92,23%),
  sendo altamente recomendado para apoiar a decisão na concessão automatizada de crédito.

Limitações do Modelo:
  1. Assume relacionamentos estritamente lineares, podendo falhar em rendas extremamente altas.
  2. Não analisa outras variáveis críticas de risco, como dívidas ativas ou dependentes.
  3. Pode prever valores negativos para perfis com scores/rendas extremamente baixos sem travas.
""")